# Fifteen-round evidence and feature inventory

New feature engineering is paused. This notebook reads saved evidence only; it never starts model fitting. Scores below are development diagnostics, not Kaggle scores. No candidate is promoted automatically.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from scripts.feature_combination_evaluation import figures, review_figures
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/feature_combination_evaluation.json").is_file())
PUBLIC = ROOT / "reports/feature_combinations"
review = json.loads((PUBLIC / "review.json").read_text())
print("Historical first-submission count:", review["first_submission"])
print("Planned candidate configurations:", len(review["plan"]))

Historical first-submission count: {'word_unigram_bigram_columns': 40000, 'explicit_similarity_columns': 8, 'total_columns': 40008, 'classifier_fits': 0, 'purpose': 'historical feature count only, not validation'}
Planned candidate configurations: 49


## Evaluation boundary

Training and query text groups are disjoint. Existing label-dependent reference banks were cross-fitted. They are **not** safe as precomputed inputs for a new CV split: inner training rows could contain statistics derived from the held-out fold. Fresh nested validation must rebuild reference pools, vocabularies, supervised weighting, screening and scaling inside each split. The current adapted training margin is in-sample; two explicit frozen-margin sensitivity models examine that dependency without claiming fresh confirmation.

In [2]:
display(pd.DataFrame(review["historical"]).sort_values(["round", "macro_auc"], ascending=[True, False]))
CHARTS = review_figures(review)

,round,study,variant,advertising_auc,legal_advice_auc,macro_auc,mean_brier,mean_log_loss,round_decision
0,1,behavioral_features,add_behavior,0.675784,0.683940,0.679862,0.231787,0.686998,DO_NOT_PROMOTE_THIS_FEATURE_SET
2,1,behavioral_features,add_scope,0.661082,0.679957,0.670520,0.236274,0.738071,DO_NOT_PROMOTE_THIS_FEATURE_SET
5,1,behavioral_features,lexical_control,0.673022,0.641993,0.657508,0.233032,0.661795,DO_NOT_PROMOTE_THIS_FEATURE_SET
9,1,behavioral_features,without_support_contrast,0.635784,0.679116,0.657450,0.245528,0.765492,DO_NOT_PROMOTE_THIS_FEATURE_SET
3,1,behavioral_features,add_support_contrast,0.629515,0.680359,0.654937,0.247833,0.734728,DO_NOT_PROMOTE_THIS_FEATURE_SET
...,...,...,...,...,...,...,...,...,...
148,15,action_window_features,qualification_windows,0.699888,0.753253,0.726571,0.244825,0.856548,DO_NOT_PROMOTE_PRIMARY
146,15,action_window_features,frozen_basic,0.693097,0.753038,0.723068,0.244398,0.824404,DO_NOT_PROMOTE_PRIMARY
147,15,action_window_features,label_null_all,0.705560,0.735817,0.720689,0.248978,0.921456,DO_NOT_PROMOTE_PRIMARY
143,15,action_window_features,answer_only,0.679254,0.760533,0.719893,0.244552,0.797983,DO_NOT_PROMOTE_PRIMARY


## Leading saved comparisons and feature dimensions

In [3]:
CHARTS[0].show(renderer="plotly_mimetype")
CHARTS[1].show(renderer="plotly_mimetype")

## Per-policy trade-offs and experiment coverage

In [4]:
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

## Probability diagnostics

In [5]:
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

## The first submission versus the new campaign

The first scored entry was lexical Version 2: at most 40,000 word unigram/bigram columns plus eight example/rule similarities. Its public/private Kaggle scores were 0.59191/0.61956. This is not the later Qwen3-4B entry (0.91808/0.91425). The exact historical vocabulary count is measured separately on the original training file; that vocabulary is never passed to the benchmark.

Round 3 policy-specific slopes are already represented by separate same-rule classifiers in this benchmark; adding an identical policy-indicator copy would not add information. R4 weighting and R5 scope are explicit alternative lexical encodings with a scope-copy control, not falsely counted as independent new signals.

In [6]:
display(pd.DataFrame(review["inventory"]))
display(pd.DataFrame(review["plan"]))
for limitation in review["limitations"]:
    print(limitation)

,fold,family,columns,training_rows,query_rows,word_columns,character_columns
0,0,local_density,12,629,234,NaN,NaN
1,0,actor_support,24,629,234,NaN,NaN
2,0,matched_pairs,12,629,234,NaN,NaN
3,0,conditioned_geometry,18,629,234,NaN,NaN
4,0,crossmodal,48,629,234,NaN,NaN
5,0,prototypes,36,629,234,NaN,NaN
6,0,consistency,48,629,234,NaN,NaN
7,0,passages,48,629,234,NaN,NaN
8,0,bm25,48,629,234,NaN,NaN
9,0,windows,48,629,234,NaN,NaN


,name,groups,kind,C,margin,lexical_mode,batch
0,all_features,"[behavior, scope, rule_alignment, relations, l...",full,1.0,adapted,plain,1
1,all_features_stronger_penalty,"[behavior, scope, rule_alignment, relations, l...",regularization,0.1,adapted,plain,1
2,anchor_refit,[],control,1.0,adapted,plain,1
3,add_behavior,[behavior],addition,1.0,adapted,plain,1
4,add_scope,[scope],addition,1.0,adapted,plain,1
5,add_rule_alignment,[rule_alignment],addition,1.0,adapted,plain,1
6,add_relations,[relations],addition,1.0,adapted,plain,1
7,add_legacy_support,[legacy_support],addition,1.0,adapted,plain,1
8,add_local_density,[local_density],addition,1.0,adapted,plain,1
9,add_matched_pairs,[matched_pairs],addition,1.0,adapted,plain,1


Development screen only: the same 881 comments informed previous research decisions.
Conditional bootstrap intervals do not undo historical selection bias or refit encoders.
Adapted training answer margins are in-sample; frozen-margin sensitivity is separate.
Do not use cached supervised banks as precomputed inputs to a new cross-validation split.
Before promotion, rebuild every supervised transform inside nested group-safe folds.
Only two policies are observed; these results do not estimate unseen-policy performance.
Early sparse variants use new same-rule training fits here, not their old pooled fits.
No new Kaggle score, independent confirmation, or automatic model promotion is produced.
